# Intro

* The disadvantages of standard RNNs
* How LSTM and GRUs work, and why they are better than standard RNNs
* A few other recent exciting developments in RNNs

## Limitations of RNNs

* Highly sensitive to recent inputs
* Low sensitivity to distant inputs
* Risk of vanishing and exploding gradients
* RNNs are great when important characteristics are a few time-steps back (like an AR model in signal processing)
* RNN-extensions give better performance: LSTM and GRU

## What is a "gate"?

$$ \sigma = \frac{1}{1+e^{-x}} $$

* Sigmoid function
* Most values are close to 0 or close to 1
* When multiplying other variables, the sigmoid can switch off (0) or on (1) the variable, thus gating information routing

## Gated Recurrent Unit (GRU)

* $r_t = \sigma(W_r[x_t;h_{t-1}])$

  * Reset Gate - Combines current input and previous hidden state to decide how much of the past to update
  * Variable $r$ is $\sim0$ or $\sim1$

* $z_t = \sigma(W_z[x_t;h_{t-1}])$

  * Update Gate - Combines current input and previous hidden state to decide what information to update (the hidden state items)
  * Variable $z$ is $\sim0$ or $\sim1$

* $n_t = \tanh(W_n[x_t;r_th_{t-1}])$

  * New information - Combines current input and previous hidden state to create a new candidate hidden state
  * Variable n is [-1,+1]
* $h_t = (1-z_t)\,n_t + z_th_{t-1}$

  * Otutput (hidden state) - Weighted combination of to-remember new state ($n$) and to-forget old state ($h_{t1}$)

## LSTM (Long-Short-Term Memory)

* $f_t = \sigma(W_f[x_t;h_{t-1}])$
  * Forget gate - Combines current input and previous hidden states to decide which previous samples to forget and which to remember.
  * Variable $f$ is $\sim 0$ or $\sim1$

* $i_t = \sigma(W_i[x_t;h_{t-1}])$
  * Input Gate - Combines current input and previous hidden states to decide which current samples samples to forget and which to remember
  * Variable $i$ is $\sim 0$ or $\sim1$

* $g_t = \tanh(W_g[x_t;h_{t-1}])$
  * Cell gate - Combines current input and previous hidden states to compute the actual representations (like "activations" in FFNs)
  * Variable $g$ ranges between -1 and +1

* $o_t = \sigma(W_o[x_t;h_{t-1}])$
  * Output gate - Combines current input and previous hidden states, and current cell states, to compute the output of the LSTM cell (hidden state)
  * Variable $o$ ranges between 0 and +1

* $c_t = f_t\,c_{t-1}+i_t\,g_t$

  * Cell state - To-remember previous cell state plus to-remember current cell state

* $h_t = o_t\,\tanh(c_t)$

* Forget the past
* Remember the present
* Plan for the future
* Recorded history

## When to use which?

* Standard RNN: Simple, fewest parameters. Good for sequences with short patterns.
* GRU: More complex than standard RNNs, but simples than LSTMs. Trains fast and highly scalable, but needs more data than RNNs.
* LSTM: Most comples, requires more training data and computation time. Explicitly includes a memory trace. Good for longer sequences or complex patterns.

## Recent Trends in RNNs

* Encoder-decoder: Like autoencoders for sequences
* Transformer: Uses "attention" (weighting by significance) and context-relevance
* Hierarchical models: Different layers see different lengths (allows for global/local interactions)
* Many others: Search for NLP - natural language processing

# LSTM

* How to create, interpret, and work with the LSTM and GRU classes in PyTorch
* That LSTM/GRU are nearly identical to the RNN class in terms of usage

## LSTM Type

In [16]:
import torch
import torch.nn as nn
import numpy as np

In [19]:
input_size  = 5
hidden_size = 3
layers_num   = 2

seq_len    = 5
batch_size = 10

In [20]:
lstm = nn.LSTM(input_size, hidden_size, layers_num)
lstm

LSTM(5, 3, num_layers=2)

In [21]:
X = torch.rand(seq_len, batch_size, input_size)

H = torch.zeros(layers_num, batch_size, hidden_size)
C = torch.zeros(layers_num, batch_size, hidden_size)

hidden_state = (H, C)

y, hidden_state = lstm(X, hidden_state)

In [23]:
print(' Input shape:', X.shape)
print('Hidden shape:', hidden_state[0].shape)
print('  Cell shape:', hidden_state[1].shape)
print('Output shape:', y.shape)

 Input shape: torch.Size([5, 10, 5])
Hidden shape: torch.Size([2, 10, 3])
  Cell shape: torch.Size([2, 10, 3])
Output shape: torch.Size([5, 10, 3])


In [33]:
for n, p in lstm.named_parameters():
  print(f'{n:>15} shape:', list(p.shape))

   weight_ih_l0 shape: [12, 5]
   weight_hh_l0 shape: [12, 3]
     bias_ih_l0 shape: [12]
     bias_hh_l0 shape: [12]
   weight_ih_l1 shape: [12, 3]
   weight_hh_l1 shape: [12, 3]
     bias_ih_l1 shape: [12]
     bias_hh_l1 shape: [12]


## DL Model Class

In [41]:
import torch
import torch.nn as nn
import numpy as np

In [55]:
class LSTMnet(nn.Module):
  def __init__(self, input_size, hidden_size, layers_num):
    super().__init__()

    self.input_size  = input_size
    self.hidden_size = hidden_size
    self.layers_num  = layers_num

    self.lstm = nn.LSTM(input_size, hidden_size, layers_num)

    self.out = nn.Linear(hidden_size, 1)

  def forward(self, x):
    H = torch.zeros(layers_num, batch_size, hidden_size)
    C = torch.zeros(layers_num, batch_size, hidden_size)

    hidden_state    = (H,C)
    y, hidden_state = self.lstm(x, hidden_state)

    o = self.out(y)

    print('Input shape: ',  x.shape)
    print('Hidden shape:', hidden_state[0].shape)
    print('Cell shape:  ',   hidden_state[1].shape)
    print('Output shape:', o.shape)

    return y, hidden_state

In [75]:
input_size  = 10
hidden_size = 5
layers_num  = 5

seq_len    = 10
batch_size = 5

net = LSTMnet(input_size, hidden_size, layers_num)

# Data
X = torch.rand(seq_len, batch_size, input_size)
y = torch.rand(seq_len, batch_size, 1)

# Without specifying hidden (default hidden)
y_hat, hidden_state = net(X)

Input shape:  torch.Size([10, 5, 10])
Hidden shape: torch.Size([5, 5, 5])
Cell shape:   torch.Size([5, 5, 5])
Output shape: torch.Size([10, 5, 1])


In [88]:
y_hat.shape

torch.Size([10, 5, 5])

In [89]:
y.shape

torch.Size([10, 5, 1])

In [87]:
loss_fn = nn.MSELoss()
loss = loss_fn(y_hat, y)
loss

tensor(0.4371, grad_fn=<MseLossBackward0>)

# GRU

In [90]:
import torch
import torch.nn as nn
import numpy as np

In [91]:
input_size  = 5
hidden_size = 10
layers_num  = 4

seq_len    = 5
batch_size = 4

In [92]:
gru = nn.GRU(input_size, hidden_size, layers_num)
gru

GRU(5, 10, num_layers=4)

In [96]:
X = torch.rand(seq_len, batch_size, input_size)
H = torch.zeros(layers_num, batch_size, hidden_size)

y, hidden_state = gru(X, H)

print(f' Input shape:', X.shape)
print(f'Hidden shape:', hidden_state.shape)
print(f'Output shape:', y.shape)

 Input shape: torch.Size([5, 4, 5])
Hidden shape: torch.Size([4, 4, 10])
Output shape: torch.Size([5, 4, 10])


In [100]:
for n, p in gru.named_parameters():
  print(f'{n:>15} - {list(p.shape)}')

   weight_ih_l0 - [30, 5]
   weight_hh_l0 - [30, 10]
     bias_ih_l0 - [30]
     bias_hh_l0 - [30]
   weight_ih_l1 - [30, 10]
   weight_hh_l1 - [30, 10]
     bias_ih_l1 - [30]
     bias_hh_l1 - [30]
   weight_ih_l2 - [30, 10]
   weight_hh_l2 - [30, 10]
     bias_ih_l2 - [30]
     bias_hh_l2 - [30]
   weight_ih_l3 - [30, 10]
   weight_hh_l3 - [30, 10]
     bias_ih_l3 - [30]
     bias_hh_l3 - [30]
